In [1]:
# Path setup

import os
import sys
from dotenv import load_dotenv

project_root = os.path.abspath("..")

if project_root not in sys.path:
    sys.path.append(project_root)

In [ ]:
# Imports

import numpy as np
from sentence_transformers import SentenceTransformer

from src.config import PDF_PATH, CHUNK_SIZE, CHUNK_OVERLAP, EMBEDDING_MODEL
from src.pdf_utils import extract_text
from src.text_utils import clean_text, chunk_text
from src.vector_store import build_index
from src.rag import retrieve
from src.embeddings import create_embeddings
from src.llm import load_model, generate_answer

In [3]:
# Load pdf

text = extract_text(PDF_PATH)
cleaned_text = clean_text(text)

chunks = chunk_text(
    cleaned_text,
    chunk_size=CHUNK_SIZE,
    overlap=CHUNK_OVERLAP
)

In [4]:
# Load model and create embeddings

model = SentenceTransformer(EMBEDDING_MODEL)

embeddings = create_embeddings(chunks, model)

index = build_index(embeddings)

In [5]:
# API Key

load_dotenv()

API_KEY = os.getenv("GEMINI_API_KEY")

llm = load_model(API_KEY)

In [6]:
# Asking a question

query = "What is an entity set?"

In [7]:
# Results - Production input (context for LLM)

context, results = retrieve(
    query=query,
    index=index,
    chunks=chunks,
    model=model
)

print(context)

 3. Entity: An Entity is a “thing” or “object” in the real world that is distinguishable from all other objects. 1. It has physical existence. 2. Each student in a college is an entity. 3. Entity can be uniquely identified. (By a primary attribute, aka Primary Key) 4. Strong Entity: Can be uniquely identified. 5. Weak Entity: Can’t be uniquely identified., depends on some other strong entity. 1. It doesn’t have suﬃcient attributes, to select a uniquely identi fiable attribute. 2. Loan -> Strong Entity, Payment -> Weak, as instalments are sequential number counter can be generated separate for each loan. 3. Weak entity depends on strong entity for existence. 4. Entity set 1. It is a set of entities of the same type that share the same properties, or a ttributes. 2. E.g., Student is an entit

some way with other entity sets. 2. Specialisation is splitting up the entity set into further sub entity sets on the basis of their functionalities, specialities and features. 3. It is a Top-Down a

In [8]:
# Results FINAL

context, results = retrieve(
    query=query,
    index=index,
    chunks=chunks,
    model=model
)

answer = generate_answer(llm, query, context)

print(answer)

An entity set is a set of entities of the same type that share the same properties or attributes.
